# Final performance review

Verifies all saved run artifacts, computes paired confidence intervals, creates comparison figures, and writes the portable review bundle.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_REPO_URL='https://github.com/selimsidan/tfm-credit-benchmark.git'
GITHUB_BRANCH='main'
DRIVE_OUTPUT_DIR='/content/drive/MyDrive/tfm_credit_benchmark'
EXPERIMENT_NAME='lending_club_900k'
FORCE_REBUILD_SPLIT=False
FORCE_RERUN=False
RUN_TESTS=True
BOOTSTRAP_RESAMPLES=2000
# ==================================================================


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys
drive.mount('/content/drive'); repo=Path('/content/tfm-credit-benchmark')
if (repo/'.git').is_dir(): subprocess.run(['git','-C',str(repo),'pull','--ff-only','origin',GITHUB_BRANCH],check=True)
else: subprocess.run(['git','clone','--branch',GITHUB_BRANCH,'--single-branch',GITHUB_REPO_URL,str(repo)],check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'],check=True)
src_path=str(repo/'src')
if src_path not in sys.path: sys.path.insert(0,src_path)
import tfmbench
print('tfmbench package:',tfmbench.__file__)
print('Repository commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())


In [ ]:
from tfmbench.config import BenchmarkConfig
from tfmbench.split import verify_manifest
if RUN_TESTS: subprocess.run([sys.executable,'-m','pytest','-q'],check=True)
config=BenchmarkConfig(DRIVE_OUTPUT_DIR,EXPERIMENT_NAME); config.validate()
if FORCE_REBUILD_SPLIT:
    from tfmbench.data import prepare_data
    prepare_data(DRIVE_OUTPUT_DIR,EXPERIMENT_NAME,force_rebuild_split=True)
manifest=verify_manifest(config.shared_dir,config); print('Verified manifest:',manifest['manifest_hash'])


In [ ]:
from tfmbench.report import build_performance_review
review=build_performance_review(DRIVE_OUTPUT_DIR,EXPERIMENT_NAME,bootstrap_resamples=BOOTSTRAP_RESAMPLES)


In [ ]:
import pandas as pd
print(review.comparison_path.read_text())
display(pd.read_csv(review.review_dir/'all_metrics.csv'))
display(review.inventory)
print('Review bundle:',review.review_dir/'review_bundle.zip')
